# Operational regression detection for CrewAI crews — Kalibra

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khan5v/kalibra/blob/main/examples/crewai/crewai_kalibra_tutorial.ipynb)

`crewai test` evaluates output quality with an LLM judge. Kalibra covers the other side: **operational regressions** — token cost, failure patterns, latency — using deterministic statistical comparison on trace data.

They solve different problems. This notebook shows two scenarios where quality scores look fine but operational metrics reveal issues that need attention.

No API key needed — runs on pre-recorded CrewAI traces.

**Stack:** [CrewAI](https://github.com/crewAIInc/crewAI) → [Phoenix](https://github.com/Arize-ai/phoenix) (traces) → [Kalibra](https://github.com/khan5v/kalibra) (statistical comparison)

In [ ]:
!pip install -q kalibra

## Setup

The traces come from a two-agent CrewAI crew (Research Analyst + Technical Writer) running 10 tasks — 5 simple, 5 complex — 4 runs per task per variant.

```
crew_run (CHAIN)
 +-- Crew.kickoff
     +-- Research Analyst._execute_core
     |   +-- LLM (Research Analyst)
     +-- Technical Writer._execute_core
         +-- LLM (Technical Writer)
```

In [ ]:
from pathlib import Path

# Find scenario trace files (local repo or current directory)
CANDIDATES = [Path("."), Path("examples/crewai")]
traces_dir = next((d for d in CANDIDATES if (d / "scenario_redistribution.jsonl").exists()), None)

if traces_dir is None:
    # Colab or standalone: download from GitHub
    import urllib.request
    traces_dir = Path(".")
    base = "https://raw.githubusercontent.com/khan5v/kalibra/main/examples/crewai"
    for name in ["scenario_redistribution.jsonl", "scenario_cost_explosion.jsonl"]:
        if not (traces_dir / name).exists():
            print(f"Downloading {name}...")
            try:
                urllib.request.urlretrieve(f"{base}/{name}", str(traces_dir / name))
            except Exception as e:
                raise FileNotFoundError(
                    f"Could not download {name}. If these files haven't been "
                    f"merged to main yet, clone the repo and run from there: "
                    f"git clone https://github.com/khan5v/kalibra && "
                    f"cd kalibra && jupyter notebook examples/crewai/crewai_kalibra_tutorial.ipynb"
                ) from e

print(f"Traces directory: {traces_dir.resolve()}")

In [ ]:
from kalibra.loader import load_traces
from kalibra.config import parse_matcher
from kalibra.engine import compare
from kalibra.renderers import render

def load_and_split(path):
    """Load a trace file and split by variant tag."""
    all_traces = load_traces(str(path))
    m_b = parse_matcher("variant == baseline")
    m_c = parse_matcher("variant == current")
    baseline = [t for t in all_traces if m_b.matches(t.metadata.get(m_b.field))]
    current  = [t for t in all_traces if m_c.matches(t.metadata.get(m_c.field))]
    print(f"Loaded {len(all_traces)} traces → {len(baseline)} baseline, {len(current)} current")
    return baseline, current

### Inspect the dataset

`kalibra inspect` shows what's in a trace file before you run a comparison — field coverage, metric readiness, and metadata values.

In [ ]:
import subprocess
print(subprocess.run(
    ["kalibra", "inspect", str(traces_dir / "scenario_redistribution.jsonl")],
    capture_output=True, text=True
).stdout)

---

## Scenario 1: Failure redistribution

You swapped the model. `crewai test` scores are stable. Every aggregate metric is flat.

| Metric | Baseline | Current | Change |
|--------|----------|---------|--------|
| Success rate | 80% | 80% | +0.0 pp |
| Tokens/trace | ~5,340 | ~5,510 | +3% (noise) |
| `crewai test` avg score | ~7.5 | ~7.4 | noise |

Everything looks identical. But **which** tasks fail shifted completely.

In [ ]:
from IPython.display import Markdown

baseline, current = load_and_split(traces_dir / "scenario_redistribution.jsonl")

result_redist = compare(
    baseline, current,
    require=["regressions <= 0"],
    baseline_source="model A",
    current_source="model B",
    # Note: metric_config is the programmatic API when skipping the config layer (vs fields.task_id in YAML)
    metric_config={"trace_breakdown": {"task_id_field": "task.id"}},
)

Markdown(render(result_redist, "markdown", verbose=True))

### What happened

Success rate: 80% → 80%. Identical. Tokens: +3%, noise.

But the per-task breakdown:
- `migration_plan` and `review_arch`: **4/4 → 0/4** — worked perfectly, now completely broken
- `compare_dbs` and `design_auth`: **0/4 → 4/4** — were broken, now fixed

The aggregate stayed flat because regressions and improvements canceled out. Without per-task comparison, you'd ship a change that breaks two task types your users depend on.

`crewai test` averages the LLM-judge scores across all tasks and sees ~7.5 in both — different tasks contributing to the same number.

---

## Scenario 2: Cost explosion

You enabled chain-of-thought reasoning. Quality improved. Time to ship?

| Metric | Baseline | Current | Change |
|--------|----------|---------|--------|
| Success rate | 75% | 90% | +15 pp |
| Tokens/trace | ~5,350 | ~7,200 | **+35%** |
| `crewai test` avg score | ~8.0 | ~8.5 | improved! |

`crewai test` says quality went up. Ship it. But it doesn't track tokens.

In [ ]:
baseline, current = load_and_split(traces_dir / "scenario_cost_explosion.jsonl")

result_cost = compare(
    baseline, current,
    require=[
        "token_delta_pct <= 20",
        "regressions <= 0",
    ],
    baseline_source="without chain-of-thought",
    current_source="with chain-of-thought",
    # Note: metric_config is the programmatic API when skipping the config layer (vs fields.task_id in YAML)
    metric_config={"trace_breakdown": {"task_id_field": "task.id"}},
)

Markdown(render(result_cost, "markdown", verbose=True))

### What happened

Success improved 75% → 90%. `crewai test` would see higher quality scores and recommend shipping.

But tokens per trace jumped ~35%. The model is generating verbose chain-of-thought output. The span breakdown confirms it: Research Analyst LLM tokens nearly doubled (+97%), Technical Writer tokens up +23%. At scale, that's a 35% cost increase — and the success rate improvement isn't even statistically significant (p=0.077).

`crewai test` has no token tracking, no cost tracking. It sees "quality improved" and moves on. Kalibra's `token_delta_pct <= 20` gate blocks the change: you're paying 35% more per trace for a marginal, statistically insignificant improvement.

---

## Side by side

| Scenario | Quality evaluation (`crewai test`) | Operational detection (Kalibra) |
|---|---|---|
| **Redistribution** | Score ~7.5 → ~7.4. No change. | 2 tasks regressed, 2 improved. **FAIL.** |
| **Cost explosion** | Score ~8.0 → ~8.5. Improved! | Tokens +35%. **FAIL.** |

### Different tools for different problems

| | Quality evaluation (`crewai test`) | Operational detection (Kalibra) |
|---|---|---|
| What it answers | "Is the output good?" | "Did cost, success, or latency change?" |
| Method | LLM-as-judge scoring | Bootstrap CIs, p-values, per-task breakdown |
| Per-task comparison | Scores per task, but averaged | Compares success rates per task across variants |
| Token / cost tracking | Not built in | Per trace and per span |
| Deterministic | No — LLM judge varies per run | Yes — pure computation |
| CI gates | Not built in | Exit code 1 on gate failure |

Quality evaluation and operational detection are complementary. `crewai test` tells you if the output reads well. Kalibra tells you if your cost tripled or if a task type that used to work now fails.

Community discussions show demand for both:
[consistent scoring](https://community.crewai.com/t/crew-how-to-keep-the-consistency-of-score/1422),
[per-task token tracking](https://github.com/crewAIInc/crewAI/issues/933),
[cost tracking per customer](https://community.crewai.com/t/how-do-you-track-llm-cost-per-customer-for-crewai-workflows/7426),
[A/B testing](https://github.com/crewAIInc/crewAI/issues/3015).
Kalibra fills the operational side of that gap.

---

## Terminal output

This is what you see when you run `kalibra compare -v` in your terminal — shown here for the redistribution scenario.

In [ ]:
print(render(result_redist, "terminal", verbose=True))

## Next steps

- **Try on your own crew.** Instrument with [Phoenix](https://github.com/Arize-ai/phoenix), export to JSONL, point Kalibra at it.
- **Gate your CI.** `kalibra compare` returns exit 1 on failure. Use [`khan5v/kalibra-action`](https://github.com/khan5v/kalibra-action) for PR comments.
- **Run `scenarios.py`** to regenerate traces or build new scenarios.

---

[Kalibra](https://github.com/khan5v/kalibra) · [Docs](https://kalibra.cc) · [GitHub Action](https://github.com/khan5v/kalibra-action) · [CrewAI](https://github.com/crewAIInc/crewAI)